# Import Librarries

In [19]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression


In [20]:

DATASET_FILE = "../eye_dataset_v2.csv"

FEATURES = [
    "left_ear",
    "right_ear",
    "ear",

    "left_eye_width",
    "left_eye_height_1",
    "left_eye_height_2",

    "right_eye_width",
    "right_eye_height_1",
    "right_eye_height_2"
]


df = pd.read_csv(DATASET_FILE)
df


,left_ear,right_ear,ear,left_eye_width,left_eye_height_1,left_eye_height_2,right_eye_width,right_eye_height_1,right_eye_height_2,label
0,0.427398,0.432328,0.429863,0.039186,0.017069,0.016427,0.035552,0.014775,0.015965,OPEN
1,0.427019,0.435539,0.431279,0.039078,0.016954,0.016420,0.035660,0.014906,0.016156,OPEN
2,0.427494,0.447180,0.437337,0.039127,0.017017,0.016436,0.035825,0.015400,0.016641,OPEN
3,0.433946,0.444521,0.439234,0.039218,0.017375,0.016662,0.035812,0.015204,0.016635,OPEN
4,0.423431,0.440666,0.432049,0.039083,0.016941,0.016157,0.036280,0.015251,0.016724,OPEN
...,...,...,...,...,...,...,...,...,...,...
995,0.156201,0.219984,0.188093,0.034444,0.005484,0.005276,0.030398,0.006173,0.007201,CLOSED
996,0.133958,0.220235,0.177097,0.034373,0.004669,0.004540,0.031537,0.006374,0.007517,CLOSED
997,0.157326,0.234718,0.196022,0.033819,0.005462,0.005179,0.030698,0.006550,0.007861,CLOSED
998,0.165653,0.212419,0.189036,0.033581,0.005775,0.005351,0.030503,0.005807,0.007151,CLOSED


## Modified Z-Score using MAD

In [21]:


def modified_z_score(series):

    median = series.median()

    mad = np.median(
        np.abs(series - median)
    )

    # Avoid division by zero
    if mad == 0:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    modified_z = (
        0.6745 *
        (series - median) /
        mad
    )

    return modified_z



### Robust Outlier Analysis

In [22]:


print("=" * 70)
print("ROBUST OUTLIER ANALYSIS")
print("=" * 70)

print()
print("Method:")
print("Modified Z-score based on Median and MAD")
print("Outlier criterion: |Modified Z| > 3.5")
print()

all_results = []

for feature in FEATURES:

    print()
    print("=" * 70)
    print(f"FEATURE: {feature}")
    print("=" * 70)

    for label in ["CLOSED", "OPEN"]:

        data = df.loc[
            df["label"] == label,
            feature
        ].dropna()

        median = data.median()

        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)

        iqr = q3 - q1

        mad = np.median(
            np.abs(data - median)
        )

        # Modified Z-score

        if mad == 0:

            modified_z = pd.Series(
                np.zeros(len(data)),
                index=data.index
            )

        else:

            modified_z = (
                0.6745 *
                (data - median) /
                mad
            )

        outlier_mask = (
            np.abs(modified_z) > 3.5
        )

        outlier_count = int(
            outlier_mask.sum()
        )

        total_count = len(data)

        outlier_percentage = (
            outlier_count /
            total_count *
            100
        )

        # Robust boundaries

        if mad == 0:

            lower_bound = median
            upper_bound = median

        else:

            lower_bound = (
                median -
                (3.5 * mad / 0.6745)
            )

            upper_bound = (
                median +
                (3.5 * mad / 0.6745)
            )


        print()
        print(f"CLASS: {label}")
        print("-" * 50)

        print(f"Count       : {total_count}")
        print(f"Median      : {median:.6f}")
        print(f"Q1          : {q1:.6f}")
        print(f"Q3          : {q3:.6f}")
        print(f"IQR         : {iqr:.6f}")
        print(f"MAD         : {mad:.6f}")

        print(
            f"Lower robust boundary : "
            f"{lower_bound:.6f}"
        )

        print(
            f"Upper robust boundary : "
            f"{upper_bound:.6f}"
        )

        print(
            f"Outliers    : "
            f"{outlier_count}"
        )

        print(
            f"Outlier %   : "
            f"{outlier_percentage:.2f}%"
        )

        # ----------------------------------------------------
        # Save result
        # ----------------------------------------------------

        all_results.append({
            "feature": feature,
            "label": label,
            "count": total_count,
            "median": median,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "MAD": mad,
            "lower_boundary": lower_bound,
            "upper_boundary": upper_bound,
            "outlier_count": outlier_count,
            "outlier_percentage": outlier_percentage
        })


# Summary Table

results_df = pd.DataFrame(
    all_results
)


print()
print()
print("=" * 70)
print("OUTLIER SUMMARY")
print("=" * 70)

print()

print(
    results_df[
        [
            "feature",
            "label",
            "outlier_count",
            "outlier_percentage"
        ]
    ].to_string(index=False)
)


# Suspicious Samples


print()
print()
print("=" * 70)
print("SUSPICIOUS SAMPLE ANALYSIS")
print("=" * 70)

print()
print(
    "Samples are listed if they are an outlier in "
    "at least one feature."
)
print()



outlier_feature_count = pd.Series(
    0,
    index=df.index,
    dtype=int
)

outlier_features = {
    index: []
    for index in df.index
}


for feature in FEATURES:

    for label in ["CLOSED", "OPEN"]:

        mask_class = (
            df["label"] == label
        )

        data = df.loc[
            mask_class,
            feature
        ]

        median = data.median()

        mad = np.median(
            np.abs(data - median)
        )

        if mad == 0:
            continue

        modified_z = (
            0.6745 *
            (data - median) /
            mad
        )

        mask_outlier = (
            np.abs(modified_z) > 3.5
        )

        outlier_indices = (
            modified_z[
                mask_outlier
            ].index
        )

        for index in outlier_indices:

            outlier_feature_count.loc[index] += 1

            outlier_features[index].append(
                feature
            )



# Create suspicious dataframe


suspicious_indices = (
    outlier_feature_count[
        outlier_feature_count > 0
    ]
    .index
)


suspicious_df = df.loc[
    suspicious_indices
].copy()


suspicious_df[
    "outlier_feature_count"
] = outlier_feature_count.loc[
    suspicious_indices
]


suspicious_df[
    "outlier_features"
] = [
    ", ".join(
        outlier_features[index]
    )
    for index in suspicious_indices
]


# Print suspicious samples


if len(suspicious_df) == 0:

    print("No robust outliers detected.")

else:

    print(
        f"Total suspicious samples: "
        f"{len(suspicious_df)}"
    )

    print()

    print(
        suspicious_df[
            [
                "label",
                "ear",
                "right_ear",
                "left_ear",
                "outlier_feature_count",
                "outlier_features"
            ]
        ]
        .sort_values(
            "outlier_feature_count",
            ascending=False
        )
        .to_string()
    )



# Specifically investigate high EAR


print()
print()
print("=" * 70)
print("HIGH EAR INVESTIGATION")
print("=" * 70)

HIGH_EAR_THRESHOLD = 0.40

high_ear = df[
    df["ear"] > HIGH_EAR_THRESHOLD
].copy()

print()
print(
    f"Samples with EAR > "
    f"{HIGH_EAR_THRESHOLD}: "
    f"{len(high_ear)}"
)

if len(high_ear) > 0:

    print()

    print(
        high_ear[
            [
                "label",
                "left_ear",
                "right_ear",
                "ear",
                "left_eye_width",
                "left_eye_height_1",
                "left_eye_height_2",
                "right_eye_width",
                "right_eye_height_1",
                "right_eye_height_2"
            ]
        ].to_string(index=True)
    )



# Specifically investigate very high right EAR


print()
print()
print("=" * 70)
print("HIGH RIGHT EAR INVESTIGATION")
print("=" * 70)

HIGH_RIGHT_EAR_THRESHOLD = 0.50

high_right_ear = df[
    df["right_ear"] > HIGH_RIGHT_EAR_THRESHOLD
].copy()

print()

print(
    f"Samples with right_ear > "
    f"{HIGH_RIGHT_EAR_THRESHOLD}: "
    f"{len(high_right_ear)}"
)

if len(high_right_ear) > 0:

    print()

    print(
        high_right_ear[
            [
                "label",
                "left_ear",
                "right_ear",
                "ear",
                "left_eye_width",
                "left_eye_height_1",
                "left_eye_height_2",
                "right_eye_width",
                "right_eye_height_1",
                "right_eye_height_2"
            ]
        ].to_string(index=True)
    )



# Save Results


results_df.to_csv(
    "robust_outlier_summary.csv",
    index=False
)

suspicious_df.to_csv(
    "suspicious_samples.csv",
    index=True
)

print()
print()
print("=" * 70)
print("ROBUST OUTLIER ANALYSIS COMPLETE")
print("=" * 70)

print()
print("Files created:")
print("  robust_outlier_summary.csv")
print("  suspicious_samples.csv")

ROBUST OUTLIER ANALYSIS

Method:
Modified Z-score based on Median and MAD
Outlier criterion: |Modified Z| > 3.5


FEATURE: left_ear

CLASS: CLOSED
--------------------------------------------------
Count       : 500
Median      : 0.139858
Q1          : 0.111112
Q3          : 0.182884
IQR         : 0.071773
MAD         : 0.032852
Lower robust boundary : -0.030611
Upper robust boundary : 0.310328
Outliers    : 34
Outlier %   : 6.80%

CLASS: OPEN
--------------------------------------------------
Count       : 500
Median      : 0.429332
Q1          : 0.414350
Q3          : 0.443593
IQR         : 0.029243
MAD         : 0.014508
Lower robust boundary : 0.354049
Upper robust boundary : 0.504616
Outliers    : 17
Outlier %   : 3.40%

FEATURE: right_ear

CLASS: CLOSED
--------------------------------------------------
Count       : 500
Median      : 0.172970
Q1          : 0.141700
Q3          : 0.221186
IQR         : 0.079486
MAD         : 0.035890
Lower robust boundary : -0.013264
Upper robust